$$
\begin{split}
\text{Bandwidth} &= \frac{\text{Transfer Rate} \times \text{Bus Width}}{8}\\
\text{Bandwidth (GB/s)} &= \frac{\text{Data Rate (Gbps)}\times \text{Bus Width (bits)} }{8}
\end{split}
$$
- 硬件配置
    - 内存控制器（mc：memory controllers），通道（channles）以及位宽
    - 内存带宽的计算
        - 64GB 的 M5 max 跟 128GB 的 M5 max内存带宽也是一致的；
        - 魔改的 48GB 版本的 4090，内存带宽跟 24GB是一致的；
- 内存带宽实测
    - GPU (torch) mps 代码
    - CPU (18 core) stream

### hardwares

```shell
system_profiler SPHardwareDataType
system_profiler SPHardwareDataType | grep "Chip\|Cores"
system_profiler SPDisplaysDataType | grep -i "Total Number of Cores"
system_profiler SPMemoryDataType
```

- 关于本机 (About This Mac) -> 更多信息... (More Info...) -> 系统报告... (System Report...)。
    - 内存
    - 图形卡/显示器 (Graphics/Displays)。
- mactop

### 内存带宽

> LPDDR5x，LPDDR5X 架构的底层 JEDEC 标准是每个最基础的数据通道（Channel）为 16-bit。

- 带宽公式跟 容量（Capacity）不直接有关；
    - Bus Width 取决于芯片设计
        - 芯片设计时，周围布有固定数量的物理通道。如果每颗 LPDDR 内存颗粒占用 64-bit 通道（4 x16），那么 512-bit 位宽就需要 8 个 64-bit 的物理通道。
        - 通道-位宽-内存die颗粒（Package，3d stacking，2个64-bit的独立工作组，2个die）
            - 4颗LPDDR5X内存颗粒围绕在GPU核心四周，但每颗颗粒内部整合了多个裸片，单颗即可提供高达128-bit的超大位宽。
    - 内存颗粒的物理体质决定了 Transfer Rate 的“理论上限”，但最终的“实际运行速率”是由芯片的内存控制器（MC）、基板电气性能与内存颗粒三者动态博弈决定的。
    - MT/s, Gbps
        - 在计算内存峰值带宽的公式中，MT/s（或 GT/s）与 Mbps（或 Gbps）在数值上是完全等价（1:1）的，但它们描述这一物理过程的视角不同。
        - MT/s 或 GT/s (Mega/Giga Transfers per second)：侧重于“物理动作的频次”
            - 每秒多少“次传输”
        - Mbps 或 Gbps (Mega/Giga bits per second)：侧重于“单车道的数据体积”
            - 每秒多少“比特”
        - $\text{bit rate} = \text{transfer rate} \times \text{bits per transfer}$
        - $\text{Gbps per pin} = \text{GT/s 或 MT/s per pin} \times \text{每次传输承载的 bit 数}$
- 残血版（32核 GPU）：对应 460 GB/s 的理论峰值内存带宽。
    - $BW_{32} = \frac{9600 \text{ MT/s} \times 384 \text{ bits}}{8 \text{ bits/Byte}} = 460.8 \text{ GB/s}$
    - 384-bit 残血版：底层架构上通常依然搭载 4 颗 物理颗粒以维持散热与封装对称性，但其中一部分通道在硅片制造阶段被激光熔断（Laser Fused），实际只有相当于 3 颗 颗粒的通道在进行有效工作（3 × 128 = 384）。
- 满血版（40核 GPU）：对应 614 GB/s 的理论峰值内存带宽。
    - $BW_{40} = \frac{9600 \text{ MT/s} \times 512 \text{ bits}}{8 \text{ bits/Byte}} = 614.4 \text{ GB/s}$
    - 512-bit 满血版：在 M5 Max 的硅片周围，对称排布着 4 颗 物理封装的内存颗粒。每颗颗粒负责 128-bit 的位宽通道（4 × 128 = 512）。
    - 512 / 16 = 32 个微观物理内存通道。拥有 32 个底层内存控制器（MC，通常在宏观上被聚合划分为 4 个巨大的内存控制簇进行统筹调度）。
    - 64GB vs. 128GB
        - 64GB 与 128GB 版本的唯一物理差异，仅仅是主板上那 4 颗 LPDDR5X 内存颗粒的单体封装密度（Density）：
        - 64GB 版本：外围对称排布着 4 颗单体容量为 16GB 的高密度颗粒（$4 \times 16\text{GB} = 64\text{GB}$）。
        - 128GB 版本：外围对称排布着 4 颗单体容量为 32GB 的超高密度颗粒（$4 \times 32\text{GB} = 128\text{GB}$）。
- 无论是 64GB 还是 128GB，M5 Max 芯片点亮的内存控制器数量（收费站）、连接的通道位宽（车道数）以及运行的内存频率（车速）完全相同。唯一改变的只是“仓库的纵深”，因此峰值带宽保持绝对一致。 
```
# mc
ioreg -p IODeviceTree -n memory | grep amcc-ctrr
```

#### 48GB魔改版本的 4090

- RTX 4090 的核心是一颗代号为 AD102 的 GPU。NVIDIA 在设计 AD102 时，赋予了它 384-bit 的显存位宽。原厂的 24GB 版本，是在 PCB 板上贴了 12 颗单颗容量为 2GB 的 GDDR6X 显存颗粒（$12 \times 32\text{-bit} = 384\text{-bit}$），运行频率为 21Gbps，从而得出 1008GB/s 的带宽。
    - AD102 全芯片有 12 个 32-bit memory controllers。
- 华强北或海外极客对 4090 进行 48GB 魔改，本质上是对物理硬件的“偷梁换柱”，通常有两种实现方式：
    - 单颗高密度替换： 将原本 2GB的 GDDR6X 颗粒全部吹下来，换上 Micron（美光）单颗容量为 4GB的颗粒。
    - Clamshell 模式（蛤壳/双面贴片模式）： 如果魔改使用了具有双面显存支持的定制 PCB 板（类似于 RTX Ada 架构专业卡的板型），则会在 GPU 的正反两面各贴 12 颗 2GB 的颗粒。在 Clamshell 模式下，正反面两颗显存会共享同一个 32-bit 通道。
- 在上述两种魔改中，GPU 内部的内存控制器（MC）没有增加，总位宽依然被 AD102 芯片的物理接口死死限制在 384-bit。魔改只提升了显存颗粒的总寻址空间，但通道的宽度与 21Gbps 的频率丝毫未变，因此其理论峰值带宽依然维持在原厂的 1008GB/s。

### 内存带宽实测

- GPU MPS 测试：GPU 视角
    - 用了 Apple Silicon 的 GPU（通过 MPS 后端）
    - Apple 标称的“40 核心 GPU”，这里的“核心”在架构上更接近 NVIDIA 的 SM（Streaming Multiprocessor，流多处理器）。它不是单个工人，而是一个“车间”
    - 根据 Apple Silicon 的架构惯例，每一个 GPU Core 内部通常包含 128 个 FP32 ALU。所以，物理层面的算力并行度是：$\text{物理并发上限} = 40 \text{ Cores} \times 128 \text{ ALUs/Core} = 5120 \text{ 个计算单元}$
- CPU 内存带宽测试工具 STREAM：CPU 视角
    - 动用了全部 18 个 CPU 核心（OpenMP），

```shell
# 初始化项目并自动生成 pyproject.toml 规范文件
uv init mps-benchmark
cd mps-benchmark

# 建立底层的虚拟隔离环境（建议使用 3.10 或 3.11 维持最佳稳定性）
uv venv --python 3.11

uv add torch numpy

uv run mps-v1.py
uv run mps-v2.py
```

```
# mps-v1.py
开始内存带宽极限压测...
张量操作耗时: 0.448 秒
应用层测算有效带宽: 535.50 GB/s
```

```
# mps-v2.py
# 不再在每轮里 synchronize()，而是整批提交后统一同步，更接近设备侧持续流式吞吐。
开始更接近真实外部内存带宽的流式压测...
recommended_max_memory: 51.84 GiB
working_set_size: 11.44 GiB
buffer_triplets: 16
iterations: 128
total_streamed_bytes: 91.55 GiB
总耗时: 0.178 秒
平均每轮: 1.390 ms
更接近外部内存的有效带宽: 552.62 GB/s
```

$$
c = a + b
$$
- 一种典型的 memory-bound 操作。
    - 计算很简单，只是加法
    - 但数据量极大，需要反复读 a、读 b、写 c
    - 因此瓶颈通常不是算力，而是内存带宽
- `device = torch.device("mps")`
    - 让 PyTorch 使用 Apple 的 Metal Performance Shaders 后端，在 Apple Silicon 的 GPU 上跑。
- size = 400_000_000
    - dtype=torch.float32，每个元素 4 字节
    - 400,000,000 * 4 = 1,600,000,000 bytes，约 1.6 GB (1.49GB)
    - 代码里创建了两个输入张量：a/b，因此仅输入就大约占：3.2 GB
    - 每次计算 c = a + b 还会生成一个输出张量 c，再占：1.6 GB
    - 合计约 4.8 GB
- 预热（warmup）作用是把一些一次性开销排除掉，比如：
    - MPS 上下文初始化
    - kernel 首次编译/调度
    - runtime 首次触发的额外准备成本

```
for _ in range(5):
    c = a + b
# 等待之前排队到 MPS 的 GPU 工作真正完成
torch.mps.synchronize()
```



#### stream

```shell
xcode-select --install

brew install llvm libomp git

git clone https://github.com/jeffhammond/STREAM.git
cd STREAM

/opt/homebrew/opt/llvm/bin/clang \
  -O3 -fopenmp \
  -DSTREAM_ARRAY_SIZE=80000000 \
  -DNTIMES=20 \
  stream.c -o stream \
  -I/opt/homebrew/opt/libomp/include \
  -L/opt/homebrew/opt/libomp/lib \
  -Wl,-rpath,/opt/homebrew/opt/libomp/lib \
  -lomp

OMP_NUM_THREADS=18 ./stream
```

```
-------------------------------------------------------------
STREAM version $Revision: 5.10 $
-------------------------------------------------------------
This system uses 8 bytes per array element.
-------------------------------------------------------------
Array size = 80000000 (elements), Offset = 0 (elements)
Memory per array = 610.4 MiB (= 0.6 GiB).
Total memory required = 1831.1 MiB (= 1.8 GiB).
Each kernel will be executed 20 times.
 The *best* time for each kernel (excluding the first iteration)
 will be used to compute the reported bandwidth.
-------------------------------------------------------------
Number of Threads requested = 18
Number of Threads counted = 18
-------------------------------------------------------------
Your clock granularity/precision appears to be 1 microseconds.
Each test below will take on the order of 3921 microseconds.
   (= 3921 clock ticks)
Increase the size of the arrays if this shows that
you are not getting at least 20 clock ticks per test.
-------------------------------------------------------------
WARNING -- The above is only a rough guideline.
For best results, please be sure you know the
precision of your system timer.
-------------------------------------------------------------
Function    Best Rate MB/s  Avg time     Min time     Max time
Copy:          384027.8     0.003450     0.003333     0.003692
Scale:         383588.8     0.003433     0.003337     0.003591
Add:           361302.2     0.005541     0.005314     0.005919
Triad:         357532.6     0.005518     0.005370     0.005893
-------------------------------------------------------------
Solution Validates: avg error less than 1.000000e-13 on all three arrays
-------------------------------------------------------------
```

- Copy ≈ 383.5 GB/s, Triad ≈ 353.8 GB/s
- STREAM 这四个 kernel 的定义大致是：
    - $\text{Copy: } a_i=b_i$
    - $\text{Scale: } a_i=q\cdot b_i$
    - $\text{Add: } a_i=b_i+c_i$
    - $\text{Triad: } a_i=b_i+q\cdot c_i$